In [1]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
import re
from sklearn.cluster import KMeans
import matplotlib.ticker as ticker
from matplotlib.ticker import MaxNLocator

sns.set_style("whitegrid")
sns.set_context("talk", font_scale=1.1)

# Set the plot style for consistency
SAVE_PATH = "plots/"
os.makedirs(SAVE_PATH, exist_ok=True)

## 1. Load data

In [2]:
replace_dict = {
    'strongsort': 'StrongSORT',
    'ocsort': 'OC-SORT',
    'bytetrack': 'ByteTrack',
    'botsort': 'BoT-SORT',
    'deepocsort': 'Deep OC-SORT',
    'imprassoc': 'ImprAssOC'
}

In [3]:
# Load FPS results for GPU (3080 Ti) and TPU from CSVs
df_gpu_320 = pd.read_csv("fps_3080ti_320_results.csv")
df_gpu_320.yolo_model = df_gpu_320.yolo_model.str.replace('.pt', '')
df_gpu_512 = pd.read_csv("fps_3080ti_512_results.csv")
df_gpu_512.yolo_model = df_gpu_512.yolo_model.str.replace('.pt', '')
df_tpu_320 = pd.read_csv("fps_tpu_320_results.csv")
df_tpu_320.yolo_model = df_tpu_320.yolo_model.str.replace("_full_integer_quant_edgetpu.tflite", '')
df_tpu_320.yolo_model = df_tpu_320.yolo_model.str.replace("../tpu_weights/v8/320/", '')
df_tpu_512 = pd.read_csv("fps_tpu_512_results.csv")
df_tpu_512.yolo_model = df_tpu_512.yolo_model.str.replace("_full_integer_quant_edgetpu.tflite", '')
df_tpu_512.yolo_model = df_tpu_512.yolo_model.str.replace("../tpu_weights/v8/512/", '')

df_gpu_320["hardware"] = "GPU"
df_gpu_320["img_size"] = 320
df_gpu_512["hardware"] = "GPU"
df_gpu_512["img_size"] = 512
df_tpu_320["hardware"] = "TPU"
df_tpu_320["img_size"] = 320
df_tpu_512["hardware"] = "TPU"
df_tpu_512["img_size"] = 512

df_fps = pd.concat([df_gpu_320, df_gpu_512, df_tpu_320, df_tpu_512], ignore_index=True)
df_fps.drop_duplicates(inplace=True)
df_fps = df_fps.drop(columns=['avg_time_per_frame'])
df_fps["reid_model"] = df_fps["reid_model"].str.split("_").str[:-1].str.join('_')
df_fps['tracker'] = df_fps['tracker'].replace(replace_dict)
print(len(df_fps))
print(df_fps.columns)
df_fps.sample(5)

3660
Index(['tracker', 'yolo_model', 'reid_model', 'object_count', 'avg_fps',
       'min_time_per_frame', 'max_time_per_frame', 'hardware', 'img_size'],
      dtype='object')


,tracker,yolo_model,reid_model,object_count,avg_fps,min_time_per_frame,max_time_per_frame,hardware,img_size
2712,StrongSORT,yolov8n,osnet_x0_5,3,2.564508,365.7,476.1,TPU,320
2609,ImprAssOC,yolov8m,osnet_ain_x1_0,5,60.154571,16.2,32.4,GPU,512
1432,StrongSORT,yolov8s,osnet_ibn_x1_0,3,70.420893,12.6,35.1,GPU,512
479,ByteTrack,yolov8n,lmbn_n,5,257.319273,3.7,5.9,GPU,320
2487,ImprAssOC,yolov8n,osnet_x0_5,3,86.074528,11.3,27.0,GPU,512


In [4]:
benchmarks = []
for name in os.listdir('.'):
    if "results_" not in name:
        continue
    fps = int(name.split('_')[1][:-3])
    df = pd.read_csv(f"{name}/results.csv")
    df['fps_bench'] = fps
    benchmarks.append(df.copy())
df_benchmarks = pd.concat(benchmarks, ignore_index=True)
df_benchmarks = df_benchmarks.drop(columns=['FPS', 'Elapsed_time', 'Status'])
df_benchmarks.rename(columns={'Tracker': 'tracker', 'YOLO Model': 'yolo_model', "REID Model": "reid_model", "ImgSz": 'img_size', "HOTA": 'hota',  "MOTA": 'mota',  "IDF1": 'idf1'}, inplace=True)

df_benchmarks["yolo_model"] = df_benchmarks["yolo_model"].str.split("_").str[0]
df_benchmarks["reid_model"] = df_benchmarks["reid_model"].str.split("_").str[:-1].str.join('_')
df_benchmarks['tracker'] = df_benchmarks['tracker'].replace(replace_dict)

df_benchmarks.drop_duplicates(inplace=True)
print(len(df_benchmarks))
print(df_benchmarks.columns)
df_benchmarks.sample(5)

2880
Index(['tracker', 'reid_model', 'yolo_model', 'img_size', 'hota', 'mota',
       'idf1', 'fps_bench'],
      dtype='object')


,tracker,reid_model,yolo_model,img_size,hota,mota,idf1,fps_bench
2796,Deep OC-SORT,osnet_x0_75,yolov8s,320,34.965,37.899,47.073,5
17,StrongSORT,osnet_x0_75,yolov8m,320,39.895,45.119,55.043,15
1201,BoT-SORT,osnet_x1_0,yolov8s,512,36.861,40.337,48.153,3
2434,ImprAssOC,lmbn_n,yolov8x,512,41.091,52.152,52.077,11
398,Deep OC-SORT,osnet_ain_x1_0,yolov8l,320,37.640,43.349,51.370,15


In [5]:
# # Create a copy of df_fps and rename 'avg_fps' to 'fps_eval'
# df_fps_copy = df_fps.copy().rename(columns={'avg_fps': 'fps_eval'})

# # Add an index column to track each row
# df_fps_copy = df_fps_copy.reset_index().rename(columns={'index': 'fps_index'})

# # Merge with df_benchmarks on the common keys; include 'img_size' if needed
# merged = pd.merge(
#     df_fps_copy,
#     df_benchmarks,
#     on=['tracker', 'yolo_model', 'reid_model', 'img_size'],
#     suffixes=('', '_bench')  # Only benchmark has 'fps'
# )

# # Compute the absolute difference between df_fps's fps_eval and benchmark's fps
# merged['diff'] = (merged['fps_eval'] - merged['fps_bench']).abs()

# # For each original df_fps row (identified by fps_index), choose the benchmark row with the smallest fps difference
# best_matches = merged.sort_values('diff').groupby('fps_index', as_index=False).first()

# # Rename the benchmark's fps column to 'fps_benсh'
# # best_matches = best_matches.rename(columns={'fps': 'fps_bench'})

# # Merge back the selected benchmark columns ('hota', 'mota', 'idf1', and 'fps_benсh') to the original df_fps_copy
# df_all = pd.merge(
#     df_fps_copy,
#     best_matches[['fps_index', 'fps_bench', 'hota', 'mota', 'idf1']],
#     on='fps_index',
#     how='left'
# )

# # Optionally, drop the temporary 'fps_index' column
# df_all = df_all.drop(columns='fps_index')

# # Now df_all is a copy of df_fps (with 'fps_eval' instead of 'avg_fps') 
# # and with added benchmark columns, where the benchmark's fps column is renamed to 'fps_benсh'
# df_all.sample(5)

# Create a copy of df_fps and rename 'avg_fps' to 'fps_eval'
df_fps_copy = df_fps.copy().rename(columns={'avg_fps': 'fps_eval'})
df_fps_copy = df_fps_copy.reset_index().rename(columns={'index': 'fps_index'})

# Merge with df_benchmarks on the common keys
merged = pd.merge(
    df_fps_copy,
    df_benchmarks,
    on=['tracker', 'yolo_model', 'reid_model', 'img_size'],
    suffixes=('', '_bench')
)

def interpolate_metrics(row, benchmarks):
    """Interpolate benchmark metrics using the fps_eval value."""
    fps_eval = row['fps_eval']
    # Sort benchmarks by fps_bench
    benchmarks = benchmarks.sort_values('fps_bench')
    # If fps_eval is below or above the available range, return boundary values
    if fps_eval <= benchmarks['fps_bench'].min():
        return benchmarks.iloc[0][['fps_bench', 'hota', 'mota', 'idf1']]
    if fps_eval >= benchmarks['fps_bench'].max():
        return benchmarks.iloc[-1][['fps_bench', 'hota', 'mota', 'idf1']]
    # Find the lower and upper benchmarks
    lower = benchmarks[benchmarks['fps_bench'] <= fps_eval].iloc[-1]
    upper = benchmarks[benchmarks['fps_bench'] >= fps_eval].iloc[0]
    # Compute interpolation factor
    t = (fps_eval - lower['fps_bench']) / (upper['fps_bench'] - lower['fps_bench'])
    interpolated = {}
    interpolated['fps_bench'] = lower['fps_bench'] + t * (upper['fps_bench'] - lower['fps_bench'])
    for metric in ['hota', 'mota', 'idf1']:
        interpolated[metric] = lower[metric] + t * (upper[metric] - lower[metric])
    return pd.Series(interpolated)

# For each original row (identified by fps_index), interpolate benchmark metrics
interp_df = merged.groupby('fps_index').apply(lambda grp: interpolate_metrics(grp.iloc[0], grp), include_groups=False).reset_index()

# Merge interpolated benchmark metrics back with df_fps_copy
df_all = pd.merge(df_fps_copy, interp_df, on='fps_index', how='left')
df_all = df_all.drop(columns='fps_index')


In [6]:
print('tracker', df_all.tracker.unique())
print('yolo_model', df_all.yolo_model.unique())
print('reid_model', df_all.reid_model.unique())
print('img_size', df_all.img_size.unique())


tracker ['StrongSORT' 'OC-SORT' 'ByteTrack' 'BoT-SORT' 'Deep OC-SORT' 'ImprAssOC']
yolo_model ['yolov8n' 'yolov8s' 'yolov8m' 'yolov8l' 'yolov8x']
reid_model ['osnet_x1_0' 'osnet_x0_75' 'osnet_x0_5' 'osnet_x0_25' 'clip' 'lmbn_n'
 'osnet_ibn_x1_0' 'osnet_ain_x1_0']
img_size [320 512]


## 2. Benchmarks analyze 

In [7]:
metrics = ['hota', 'mota', 'idf1']

### Metric = Metric(yolo_size)

In [8]:
yolo_models = df_all.yolo_model.unique()  # Order as in the dataframe or set a desired order
ordered_yolo = ['yolov8n', 'yolov8s', 'yolov8m', 'yolov8l', 'yolov8x']
img_sizes = df_all.img_size.unique()
trackers = df_all.tracker.unique()
metrics = ['hota', 'mota', 'idf1']

cur_save_path = SAVE_PATH + "yolo_size_vs_metric/"
os.makedirs(cur_save_path, exist_ok=True)

for tracker in trackers:
    # Create a grid: rows = metrics (3), columns = image sizes
    fig, axes = plt.subplots(nrows=len(metrics), ncols=len(img_sizes), figsize=(18, 12), 
                             sharex=True, sharey='row')
    
    for row, metric in enumerate(metrics):
        for col, img_size in enumerate(img_sizes):
            ax = axes[row, col]
            # Filter and group by YOLO model for the given tracker and image size.
            grouped = df_benchmarks[
                (df_benchmarks.img_size == img_size) &
                (df_benchmarks.tracker == tracker) &
                (df_benchmarks.fps_bench == 30)
            ].groupby(['yolo_model'])[metrics].mean().reset_index()
            
            grouped['yolo_model'] = pd.Categorical(grouped['yolo_model'], categories=ordered_yolo, ordered=True)
            grouped = grouped.sort_values('yolo_model')
            
            # Plot using YOLO model as categorical x-axis.
            sns.lineplot(
                data=grouped,
                x='yolo_model',
                y=metric,
                marker='o',
                sort=False,  # maintain order in grouped data
                ax=ax
            )
            
            # Titles and axis labels
            if row == 0:
                ax.set_title(f'Размер: {img_size}x{img_size}', fontsize=30)
            if row == len(metrics) - 1:
                ax.set_xlabel('Версия YOLO', fontsize=35)
            else:
                ax.set_xlabel('')
            ax.set_ylabel(metric.upper(), fontsize=30)
            ax.tick_params(axis='both', which='major', labelsize=25)
            
            # Set x-ticks: positions 0,1,2,... corresponding to YOLO model names.
            unique_ticks = grouped['yolo_model'].tolist()
            ax.set_xticks(range(len(unique_ticks)))
            ax.set_xticklabels(unique_ticks, rotation=30, fontsize=25)
            ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
            ax.grid(True, which='both', linestyle='--', linewidth=0.75)
    
    fig.tight_layout(rect=[0, 0.08, 1, 1])
    fig.savefig(cur_save_path + f'{tracker}.png')
    plt.close(fig)


In [9]:
# Define the desired orders.
ordered_yolo_subset = ['yolov8n', 'yolov8x']
ordered_img_sizes = sorted(df_all.img_size.unique())  # e.g., [320, 512]

cur_save_path = os.path.join(SAVE_PATH, "yolo_size_vs_metric")
os.makedirs(cur_save_path, exist_ok=True)

for metric in metrics:
    # Filter the data for only the two YOLO models and select only necessary columns.
    df_subset = df_benchmarks[df_benchmarks.yolo_model.isin(ordered_yolo_subset)][['tracker', 'img_size', 'yolo_model', metric]].copy()
    
    # Group by tracker, image size, and YOLO model; take the maximum value for the metric.
    grouped = df_subset.groupby(['tracker', 'img_size', 'yolo_model'])[metric].mean().reset_index()
    
    # Force the ordering of yolo_model.
    grouped['yolo_model'] = pd.Categorical(grouped['yolo_model'], categories=ordered_yolo_subset, ordered=True)
    grouped = grouped.sort_values(['tracker', 'img_size', 'yolo_model'])
    
    # Pivot the table so that rows are trackers and columns are a MultiIndex: (img_size, yolo_model).
    pivot_table = grouped.pivot(index='tracker', columns=['img_size', 'yolo_model'], values=metric)
    
    # Ensure columns are ordered as desired.
    col_index = pd.MultiIndex.from_product([ordered_img_sizes, ordered_yolo_subset], names=['img_size', 'yolo_model'])
    pivot_table = pivot_table.reindex(columns=col_index)
    
    # Round numeric values to one decimal and replace missing values with a dash.
    pivot_table = pivot_table.round(1).fillna("-")
    
    # Convert the pivot table to LaTeX code.
    col_format = "|" + "l|" + "cc|" * len(ordered_img_sizes)
    col_format = None
    latex_code = pivot_table.to_latex(multicolumn=True, multirow=True, escape=True, na_rep="-", float_format="%.1f", column_format=col_format)
    
    # Replace header names if needed.
    latex_code = latex_code.replace("tracker", "Алгоритм")
    latex_code = latex_code.replace("yolo_model", "Детектор")
    latex_code = latex_code.replace("img_size", "Размер изображения")
    
    # --- Post-process the header to merge pairs of columns for each image size ---
    # First, merge two adjacent \multicolumn{1}{c}{<img_size>} entries into one \multicolumn{2}{c}{<img_size>}
    lines = latex_code.splitlines()
    new_lines = []
    for line in lines:
        new_line = re.sub(r'(\\multicolumn\{1\}\{c\}\{(\d+)\})\s*&\s*(\\multicolumn\{1\}\{c\}\{\2\})', r'\\multicolumn{2}{c}{\2}', line)
        new_lines.append(new_line)
    # Now, find the index of the first occurrence of "\midrule"
    try:
        i_mid = next(i for i, line in enumerate(new_lines) if line.strip() == "\\midrule")
    except StopIteration:
        i_mid = 3  # fallback, if not found
    
    # Build custom header rows.
    header1 = " & " + " & ".join([f"\\multicolumn{{2}}{{c}}{{{img_size}}}" for img_size in ordered_img_sizes]) + " \\\\"
    header2 = " "  # empty cell for the leftmost column (tracker names)
    for img_size in ordered_img_sizes:
        for yolo in ordered_yolo_subset:
            header2 += f" & {yolo}"
    header2 += " \\\\"
    # Replace all lines between \toprule and \midrule with our header rows.
    new_lines = new_lines[:1] + [header1, header2, "\\midrule"] + new_lines[i_mid+1:]
    
    latex_code_modified = "\n".join(new_lines)
    # Replace any remaining {r} with {c} if necessary.
    latex_code_modified = latex_code_modified.replace("{r}", "{c}")
    # -----------------------------------------
    
    caption = f"Среднее значение метрики {metric.upper()} для yolov8n и yolov8x"
    label = f"tab:mean_{metric}_yolo_size"
    
    # Wrap the LaTeX table in a table environment with caption and label.
    full_latex = (
        "\\begin{table}[htbp]\n"
        f"\n\\caption{{{caption}}}\n" +
        f"\\label{{{label}}}\n" +
        "\\centering\n" +
        latex_code_modified +
        "\\end{table}"
    )
    
    # Save the LaTeX code to a file.
    filename = f"mean_{metric}_yolo_size.tex"
    full_latex = full_latex.replace("{r}", "{c}")
    with open(os.path.join(cur_save_path, filename), "w") as f:
        f.write(full_latex)
    
    print(f"Saved LaTeX table for {metric} as {filename}")

Saved LaTeX table for hota as mean_hota_yolo_size.tex
Saved LaTeX table for mota as mean_mota_yolo_size.tex
Saved LaTeX table for idf1 as mean_idf1_yolo_size.tex


### Metric = Metric(yolo size, reid_model)

In [10]:
yolo_models = df_all.yolo_model.unique()
# yolo_models = ["yolov8n", "yolov8s", "yolov8x"]
img_sizes = df_all.img_size.unique()
trackers = df_all.tracker.unique()
reid_models = df_all.reid_model.unique()
reid_models = [x for x in reid_models if 'ibn' not in x and 'ain' not in x]
reid_models = sorted(reid_models, key=lambda x: len(x), reverse=True)

cur_save_path = SAVE_PATH + "yolo_size_and_reid_vs_metric/"
os.makedirs(cur_save_path, exist_ok=True)

for tracker in trackers:
    # Create a grid: rows = metrics, columns = image sizes
    fig, axes = plt.subplots(nrows=len(metrics), ncols=len(img_sizes), figsize=(18, 12),
                             sharex=True, sharey='row')
    
    for row, metric in enumerate(metrics):
        for col, img_size in enumerate(img_sizes):
            ax = axes[row, col]
            # Loop over each YOLO model to plot its line
            for yolo_model in yolo_models:
                # Group by 'reid_model' (the x-axis now) and compute mean for the given metrics.
                grouped = df_benchmarks[
                    (df_benchmarks.img_size == img_size) &
                    (df_benchmarks.yolo_model == yolo_model) &
                    (df_benchmarks.tracker == tracker) & 
                    (df_benchmarks.fps_bench == 30) & 
                    (df_benchmarks.reid_model.isin(reid_models))
                ].groupby(['reid_model'])[metrics].mean().reset_index()
                
                sns.lineplot(
                    data=grouped,
                    x='reid_model',
                    y=metric,
                    marker='o',
                    label=yolo_model,
                    ax=ax
                )
            # Titles and labels
            if row == 0:
                ax.set_title(f'Размер: {img_size}x{img_size}', fontsize=30)
            if row == len(metrics) - 1:
                ax.set_xlabel('ReID Model', fontsize=35)
            else:
                ax.set_xlabel('')
            ax.set_ylabel(metric.upper(), fontsize=30)
            ax.tick_params(axis='both', which='major', labelsize=25)

            # For categorical x-axis, we assign positions 0,1,2,...
            ax.set_xticks(range(len(reid_models)))
            ax.set_xticklabels(reid_models, rotation=30, fontsize=25)
            ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
            ax.grid(True, which='both', linestyle='--', linewidth=0.75)
    
    # Remove individual legends from subplots
    handles, labels = axes[0, 0].get_legend_handles_labels()
    for ax in axes.flatten():
        if ax.get_legend() is not None:
            ax.get_legend().remove()
    
    # Add one common legend at the bottom center.
    fig.legend(handles, labels, loc='lower center', ncol=len(yolo_models),
               title='Детектор', fontsize=30, title_fontsize=30)
    
    fig.tight_layout(rect=[0, 0.12, 1, 1])
    fig.savefig(cur_save_path + f'{tracker}.png')
    plt.close(fig)

In [11]:
# Filter and sort ReID models.
reid_models = df_all.reid_model.unique()
reid_models = [x for x in reid_models if 'ibn' not in x and 'ain' not in x]
reid_models = sorted(reid_models, key=lambda x: len(x), reverse=True)

# Use only the HOTA metric.
metrics = ["hota"]

# Use the given YOLO models.
yolo_models = ["yolov8n", "yolov8s"]

# Sort image sizes and select the smallest and largest for subplots.
img_sizes_sorted = sorted(df_benchmarks.img_size.unique())
selected_img_sizes = [img_sizes_sorted[0], img_sizes_sorted[-1]]

# Get unique fps_bench values and trackers.
fps_values = sorted(df_benchmarks.fps_bench.unique())
trackers = sorted(df_benchmarks.tracker.unique())

# Set the save path.
cur_save_path = SAVE_PATH + "heatmap_metrics_vs_tracker_reid/"
os.makedirs(cur_save_path, exist_ok=True)

# Loop over each combination of YOLO, image size, and fps_bench.
for yolo in yolo_models:
    for img_size in selected_img_sizes:
        for fps in fps_values:
            # Filter data for current combination.
            df_subset = df_benchmarks[
                (df_benchmarks.yolo_model == yolo) &
                (df_benchmarks.img_size == img_size) &
                (df_benchmarks.fps_bench == fps)
            ]
            # Skip if no data.
            if df_subset.empty:
                continue
            
            # Create a figure with one subplot (only HOTA).
            fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(12, 12))
            
            # Pivot: rows = reid_model, columns = tracker, value = mean HOTA.
            pivot = df_subset.pivot_table(index="reid_model", columns="tracker", values="hota", aggfunc="mean")
            pivot = pivot.reindex(reid_models)
            
            sns.heatmap(pivot, annot=True, fmt=".1f", cmap="viridis", ax=ax, cbar=True, annot_kws={"size": 30})
            ax.set_title(f"HOTA, {yolo}, {img_size}x{img_size}, {fps} FPS", fontsize=30)
            ax.set_xlabel("Method", fontsize=30)
            ax.set_ylabel("ReID Model", fontsize=30)
            ax.xaxis.set_major_locator(MaxNLocator(integer=True))
            ax.yaxis.set_major_locator(MaxNLocator(integer=True))
            
            # Set fixed tick positions and labels.
            ax.set_xticks(np.arange(len(pivot.columns)) + 0.5)
            ax.set_xticklabels(pivot.columns, rotation=45, ha="right", fontsize=30)
            ax.set_yticks(np.arange(len(pivot.index)) + 0.5)
            ax.set_yticklabels(pivot.index, rotation=45, fontsize=30)
            
            # fig.suptitle(f"{yolo}, {img_size}x{img_size}, {fps} FPS", fontsize=30)
            fig.tight_layout(rect=[0, 0.01, 1, 1])
            filename = f"heatmap_metrics_{yolo}_size{img_size}_fps{fps}.png"
            fig.savefig(os.path.join(cur_save_path, filename))
            plt.close(fig)

### Metric of method = Metric(FPS Benchmark, yolo size)

In [12]:
fps_data = df_fps[df_fps.avg_fps < 16].avg_fps.values.reshape(-1, 1)
n_clusters = 5 
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
kmeans.fit_predict(fps_data)
print(sorted(kmeans.cluster_centers_.reshape(-1).astype(int)))

[1, 3, 5, 11, 14]


In [13]:
metrics = ['hota', 'mota', 'idf1']
yolo_models = ["yolov8n", "yolov8s", "yolov8x"]
img_sizes = df_all.img_size.unique()
trackers = df_all.tracker.unique()

cur_save_path = SAVE_PATH + "fps_vs_metric/"
os.makedirs(cur_save_path, exist_ok=True)

for tracker in trackers:
    # Create a 3 (rows: metrics) x 2 (cols: image sizes) grid
    fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(20, 10), sharex=True, sharey='row')
    
    for row, metric in enumerate(metrics):
        for col, img_size in enumerate(img_sizes):
            ax = axes[row, col]
            for yolo_model in yolo_models:
                grouped = df_benchmarks[
                    (df_benchmarks.img_size == img_size) &
                    (df_benchmarks.yolo_model == yolo_model) &
                    (df_benchmarks.tracker == tracker)
                ].groupby(['fps_bench'])[metrics].mean().reset_index()
                
                sns.lineplot(
                    data=grouped,
                    x='fps_bench',
                    y=metric,
                    marker='o',
                    label=yolo_model,
                    ax=ax
                )
            if row == 0:
                ax.set_title(f'Размер: {img_size}x{img_size}', fontsize=30)
            if row == 2:
                ax.set_xlabel('Частота кадров', fontsize=30)
            else:
                ax.set_xlabel('')
            ax.set_ylabel(metric.upper(), fontsize=30)
            ax.tick_params(axis='both', which='major', labelsize=25)
            
            # Set x-ticks based on the unique fps_bench values for the current tracker and img_size.
            unique_ticks = sorted(
                df_benchmarks[
                    (df_benchmarks.img_size == img_size) &
                    (df_benchmarks.tracker == tracker)
                ]['fps_bench'].unique()
            )
            ax.set_xticks(unique_ticks)
            ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
            
            # Enable a dense grid
            ax.grid(True, which='both', linestyle='--', linewidth=0.75)
    
    # Remove individual legends from each subplot.
    handles, labels = axes[0, 0].get_legend_handles_labels()
    for ax in axes.flatten():
        if ax.get_legend() is not None:
            ax.get_legend().remove()
    
    # Add a single common legend at the bottom center.
    fig.legend(handles, labels, loc='lower center', ncol=len(yolo_models), title='Детектор',
               fontsize=30, title_fontsize=30)
    
    # fig.suptitle(f'Влияние частоты кадров на метрики для алгоритма {tracker}', fontsize=24, y=0.98)
    fig.tight_layout(rect=[0, 0.12, 1, 1])
    fig.savefig(cur_save_path + f'{tracker}.png')
    plt.close(fig)

### FPS_EVAL = FPS_EVAL(object_count)

GPU vs TPU

In [14]:
from matplotlib.lines import Line2D

# Assume df_all is your DataFrame (with 3660 rows and 13 columns)
# and SAVE_PATH is already defined.
yolo_models = ["yolov8n", "yolov8s"]
ordered_yolo = yolo_models  # desired order

# Sort image sizes and select the smallest and largest for subplots.
img_sizes_sorted = sorted(df_all.img_size.unique())
selected_img_sizes = [img_sizes_sorted[0], img_sizes_sorted[-1]]
trackers = df_all.tracker.unique()

cur_save_path = SAVE_PATH + "fps_vs_object_count/"
os.makedirs(cur_save_path, exist_ok=True)

# Define dash mapping for hardware (used in the plot).
dash_mapping = {"GPU": "", "TPU": (2,2)}
hardware_order = ["GPU", "TPU"]

# Get the default color palette used by seaborn for lineplot.
palette = sns.color_palette(n_colors=len(ordered_yolo))

# Define marker styles for hardware legend entries (only markers, no lines).
hardware_markers = {"GPU": "o", "TPU": "x"}  # e.g., circle for GPU, square for TPU

for tracker in trackers:
    # Create a figure with 1 row and 2 columns (for smallest and largest image sizes).
    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(18, 6), sharex=True, sharey=True)
    
    for col, img_size in enumerate(selected_img_sizes):
        ax = axes[col]
        # Filter data for the current tracker, image size, and only selected YOLO models.
        data = df_all[
            (df_all.img_size == img_size) &
            (df_all.tracker == tracker) &
            (df_all.yolo_model.isin(yolo_models))
        ]
        # Group by yolo_model, object_count, and hardware, averaging fps_eval.
        grouped = data.groupby(['yolo_model', 'object_count', 'hardware'])['fps_eval'].mean().reset_index()
        grouped['yolo_model'] = pd.Categorical(grouped['yolo_model'], categories=ordered_yolo, ordered=True)
        grouped = grouped.sort_values(['yolo_model', 'object_count'])
        
        # Plot fps_eval vs. object_count with separate lines for each combination.
        sns.lineplot(
            data=grouped,
            x='object_count',
            y='fps_eval',
            hue='yolo_model',
            style='hardware',
            markers=True,
            dashes=dash_mapping,
            ax=ax,
            palette=palette
        )
        
        # Set x-axis ticks to the sorted unique object_count values.
        unique_obj_counts = sorted(grouped["object_count"].unique())
        ax.set_xticks(unique_obj_counts)
        
        ax.set_title(f'Image Size: {img_size}x{img_size}', fontsize=20)
        ax.set_xlabel('Object Count', fontsize=15)
        ax.set_ylabel('FPS Eval', fontsize=15)
        ax.tick_params(axis='both', labelsize=12)
        ax.grid(True, linestyle='--', linewidth=0.75)
        
        # Remove individual legend from each subplot.
        ax.get_legend().remove()
    
    # Create custom legend handles.
    # For YOLO models: show colored lines only (no markers).
    hue_handles = [
        Line2D([], [], color=palette[i], lw=2, linestyle='-', marker=None, label=ym)
        for i, ym in enumerate(ordered_yolo)
    ]
    # For hardware: show only a marker in black (no connecting line).
    hardware_handles = [
        Line2D([], [], color='black', marker=hardware_markers[hw], linestyle='None', markersize=10, label=hw)
        for hw in hardware_order
    ]
    # Combine the two groups of handles.
    all_handles = hue_handles + hardware_handles
    # Create a combined legend below the subplots.
    fig.legend(all_handles, [h.get_label() for h in all_handles],
               loc='lower center', ncol=len(ordered_yolo) + len(hardware_order), fontsize=12)
    
    fig.tight_layout(rect=[0, 0.1, 1, 0.95])
    fig.savefig(cur_save_path + f'{tracker}_gpu_tpu.png')
    plt.close(fig)

In [15]:
# Assume df_all is your DataFrame (with 3660 rows and 13 columns)
# and SAVE_PATH is already defined.
yolo_models = ["yolov8n", "yolov8s"]
ordered_yolo = yolo_models  # desired order

# Sort image sizes and select the smallest and largest for subplots.
img_sizes_sorted = sorted(df_all.img_size.unique())
selected_img_sizes = [img_sizes_sorted[0], img_sizes_sorted[-1]]
trackers = df_all.tracker.unique()

cur_save_path = SAVE_PATH + "fps_vs_object_count/"
os.makedirs(cur_save_path, exist_ok=True)

# Get the default color palette used by seaborn for lineplot.
palette = sns.color_palette(n_colors=len(ordered_yolo))

# Create a figure with one row per tracker and 2 columns for the two image sizes.
fig, axes = plt.subplots(nrows=len(trackers), ncols=2, figsize=(18, 6 * len(trackers)), sharex=True, sharey="row")

for i, tracker in enumerate(trackers):
    for j, img_size in enumerate(selected_img_sizes):
        ax = axes[i, j] if len(trackers) > 1 else axes[j]
        # Filter data for the current tracker, image size, only selected YOLO models, and only TPU hardware.
        data = df_all[
            (df_all.img_size == img_size) &
            (df_all.tracker == tracker) &
            (df_all.yolo_model.isin(yolo_models)) &
            (df_all.hardware == "TPU")
        ]
        # Group by yolo_model and object_count, averaging fps_eval.
        grouped = data.groupby(['yolo_model', 'object_count'])['fps_eval'].mean().reset_index()
        grouped['yolo_model'] = pd.Categorical(grouped['yolo_model'], categories=ordered_yolo, ordered=True)
        grouped = grouped.sort_values(['yolo_model', 'object_count'])
        
        # Plot fps_eval vs. object_count with separate lines for each YOLO model.
        sns.lineplot(
            data=grouped,
            x='object_count',
            y='fps_eval',
            hue='yolo_model',
            markers=True,
            ax=ax,
            palette=palette
        )
        
        # Set x-axis ticks to the sorted unique object_count values.
        unique_obj_counts = sorted(grouped["object_count"].unique())
        ax.set_xticks(unique_obj_counts)
        
        ax.set_title(f'{tracker} - Image Size: {img_size}x{img_size}', fontsize=30)
        ax.set_xlabel('Object Count', fontsize=30)
        ax.set_ylabel('FPS Eval', fontsize=30)
        ax.tick_params(axis='both', labelsize=25)
        ax.grid(True, linestyle='--', linewidth=0.75)
        
        # Remove the individual legend from each subplot.
        ax.get_legend().remove()

# Create custom legend handles for YOLO models: colored lines (no markers).
hue_handles = [
    Line2D([], [], color=palette[i], lw=2, linestyle='-', marker=None, label=ym)
    for i, ym in enumerate(ordered_yolo)
]

# Create a combined legend below the subplots.
fig.legend(hue_handles, [h.get_label() for h in hue_handles],
           loc='lower center', ncol=len(ordered_yolo), fontsize=30)

fig.tight_layout(rect=[0, 0.04, 1, 1])
fig.savefig(cur_save_path + "all_trackers_tpu.png")
plt.close(fig)

Heatmap

In [16]:
reid_models = df_all.reid_model.unique()
reid_models = [x for x in reid_models if 'ibn' not in x and 'ain' not in x]
reid_models = sorted(reid_models, key=lambda x: len(x), reverse=True)

# Get unique trackers and yolo models.
trackers = sorted(df_all.tracker.unique())
yolo_models_all = sorted(df_all.yolo_model.unique())
yolo_models = ["yolov8n", "yolov8s"]
img_sizes = sorted(df_all.img_size.unique())

# We'll create a plot for each combination of YOLO and image size.
cur_save_path = SAVE_PATH + "heatmap_fps_vs_tracker_reid/"
os.makedirs(cur_save_path, exist_ok=True)

# Filter for TPU only.
df_tpu = df_all[df_all.hardware == "TPU"]

for yolo in yolo_models:
    for img_size in img_sizes:
        # Filter data for the current yolo model and image size.
        df_subset = df_tpu[(df_tpu.yolo_model == yolo) & (df_tpu.img_size == img_size)]
        
        # Create a pivot table: rows = reid_model, columns = tracker, values = mean fps_eval.
        pivot = df_subset.pivot_table(index="reid_model", columns="tracker", values="fps_eval", aggfunc="mean")
        
        # Reindex the rows to match the sorted reid_models list.
        pivot = pivot.reindex(reid_models)
        
        # Skip plotting if no data exists.
        if pivot.empty:
            continue
        
        plt.figure(figsize=(12, 8))
        sns.heatmap(pivot, annot=True, fmt=".1f", cmap="viridis", center=4, vmax=30)
        plt.title(f"FPS (eval) for TPU - YOLO: {yolo}, Image Size: {img_size}", fontsize=16)
        plt.xlabel("Tracker", fontsize=14)
        plt.ylabel("ReID Model", fontsize=14)
        plt.tight_layout()
        filename = f"heatmap_fps_{yolo}_size{img_size}.png"
        plt.savefig(os.path.join(cur_save_path, filename))
        plt.close()


In [17]:
# Filter and sort ReID models.
reid_models = df_all.reid_model.unique()
reid_models = [x for x in reid_models if 'ibn' not in x and 'ain' not in x]
reid_models = sorted(reid_models, key=lambda x: len(x), reverse=True)

# Get unique trackers and YOLO models.
trackers = sorted(df_all.tracker.unique())
yolo_models_all = sorted(df_all.yolo_model.unique())
yolo_models = ["yolov8n", "yolov8s"]
img_sizes = sorted(df_all.img_size.unique())

# Set the save path.
cur_save_path = SAVE_PATH + "heatmap_fps_hota_vs_tracker_reid/"
os.makedirs(cur_save_path, exist_ok=True)

# Filter for TPU only.
df_tpu = df_all[df_all.hardware == "TPU"]

for yolo in yolo_models:
    for img_size in img_sizes:
        # Filter data for the current YOLO model and image size.
        df_subset = df_tpu[(df_tpu.yolo_model == yolo) & (df_tpu.img_size == img_size)]
        
        # Create pivot tables:
        # Left heatmap: mean fps_eval
        pivot_fps = df_subset.pivot_table(index="reid_model", columns="tracker", values="fps_eval", aggfunc="mean")
        # Right heatmap: mean HOTA
        pivot_hota = df_subset.pivot_table(index="reid_model", columns="tracker", values="hota", aggfunc="mean")
        
        # Reindex the rows to match the sorted ReID models.
        pivot_fps = pivot_fps.reindex(reid_models)
        pivot_hota = pivot_hota.reindex(reid_models)
        
        # Skip plotting if no data exists.
        if pivot_fps.empty or pivot_hota.empty:
            continue
        
        # Create a figure with two subplots side by side.
        fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(16, 12), sharex=True)
        
        # Left heatmap: FPS
        sns.heatmap(pivot_fps, annot=True, fmt=".1f", cmap="viridis", center=4, vmax=30, ax=axes[0])
        axes[0].set_title(f"FPS (eval) for TPU - YOLO: {yolo}, Image Size: {img_size}", fontsize=16)
        # axes[0].set_xlabel("Tracker", fontsize=14)
        axes[0].set_xlabel("", fontsize=14)
        axes[0].set_ylabel("", fontsize=14)
        
        # Right heatmap: HOTA
        sns.heatmap(pivot_hota, annot=True, fmt=".1f", cmap="viridis", ax=axes[1])
        axes[1].set_title(f"HOTA for TPU - YOLO: {yolo}, Image Size: {img_size}", fontsize=16)
        # axes[1].set_xlabel("Tracker", fontsize=14)
        axes[1].set_ylabel("")  # shared y-axis
        axes[0].set_xlabel("", fontsize=14)
        
        
        fig.tight_layout(rect=[0, 0.03, 1, 0.95])
        filename = f"heatmap_fps_hota_{yolo}_size{img_size}.png"
        fig.savefig(os.path.join(cur_save_path, filename))
        plt.close(fig)

Correlation stuff

In [36]:
metrics = ["mota", "hota", "idf1"]

# Compute the correlation matrix using the appropriate DataFrame (e.g., df_benchmarks).
corr_df = df_benchmarks[metrics].corr()

# Round the values to two decimal places.
corr_df = corr_df.round(2)

# Convert the correlation DataFrame to LaTeX code.
latex_code = corr_df.to_latex(multicolumn=True, multirow=True, escape=True, float_format="%.2f")

# Optionally, replace header names if needed.
latex_code = latex_code.replace("mota", "MOTA")
latex_code = latex_code.replace("hota", "HOTA")
latex_code = latex_code.replace("idf1", "IDF1")

# Set caption and label.
caption = "Correlation between tracking performance metrics."
label = "tab:correlation_metrics"

# Wrap the LaTeX table in a table environment with caption and label.
full_latex = (
    "\\begin{table}[htbp]\n"
    "\\centering\n"
    f"\\caption{{{caption}}}\n" +
    f"\\label{{{label}}}\n" +
    latex_code +
    "\\end{table}"
)

# Set the save path.
cur_save_path = os.path.join(SAVE_PATH, "correlations")
os.makedirs(cur_save_path, exist_ok=True)
filename = "metrics.tex"
with open(os.path.join(cur_save_path, filename), "w") as f:
    f.write(full_latex)

print(f"Saved correlation table as {filename}")


Saved correlation table as metrics.tex


In [27]:
df_all.info(), df_all.tracker.unique()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3660 entries, 0 to 3659
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   tracker             3660 non-null   object 
 1   yolo_model          3660 non-null   object 
 2   reid_model          3660 non-null   object 
 3   object_count        3660 non-null   int64  
 4   fps_eval            3660 non-null   float64
 5   min_time_per_frame  3660 non-null   float64
 6   max_time_per_frame  3660 non-null   float64
 7   hardware            3660 non-null   object 
 8   img_size            3660 non-null   int64  
 9   fps_bench           3660 non-null   float64
 10  hota                3660 non-null   float64
 11  mota                3660 non-null   float64
 12  idf1                3660 non-null   float64
dtypes: float64(7), int64(2), object(4)
memory usage: 371.8+ KB


(None,
 array(['StrongSORT', 'OC-SORT', 'ByteTrack', 'BoT-SORT', 'Deep OC-SORT',
        'ImprAssOC'], dtype=object))

In [ ]:
import os
import pandas as pd

# Filter the data for TPU only.
df_tpu = df_all[df_all.hardware == "TPU"]

# For each unique method (tracker), compute the correlation between fps_eval and object_count.
corr_list = []
for method, group in df_tpu.groupby("tracker"):
    # Compute Pearson correlation between fps_eval and object_count.
    corr_val = group["fps_eval"].corr(group["object_count"])
    corr_list.append({"Method": method, "Correlation (fps_eval vs object_count)": corr_val})

# Create a DataFrame from the correlation results.
corr_df = pd.DataFrame(corr_list)
corr_df = corr_df.sort_values("Method")
corr_df["Correlation (fps_eval vs object_count)"] = corr_df["Correlation (fps_eval vs object_count)"].round(2)

# Convert the DataFrame to LaTeX.
latex_code = corr_df.to_latex(index=False, float_format="%.2f")

# Set the save path and save the LaTeX table.
latex_code = latex_code.replace("Method", "Алгоритм")
latex_code = latex_code.replace("Correlation (fps_eval vs object_count)", "Корреляция")

caption = "Корреляция частоты работы с количеством объектов на изображении"
label = "tab:correlation_fps_object"

full_latex = (
    "\\begin{table}[htbp]\n"
    f"\\caption{{{caption}}}\n"
    f"\\label{{{label}}}\n"
    "\\centering\n"
    f"{latex_code}\n"
    "\\end{table}"
)


cur_save_path = os.path.join(SAVE_PATH, "correlations")
os.makedirs(cur_save_path, exist_ok=True)
filename = "correlation_fps_objectcount.tex"
with open(os.path.join(cur_save_path, filename), "w") as f:
    f.write(full_latex)

print(f"Saved LaTeX table as {filename}")


Saved LaTeX table as correlation_fps_objectcount.tex


In [102]:
import os
import pandas as pd
import numpy as np

# Filter for TPU only.
df_tpu =df_benchmarks

# Define bins and labels for fps_bench intervals.
bins = [0, 6, float('inf')]
# Labels that include both the left and right edge.
labels = ["(0, 6]", "[6, 30]"]

# Create a new column for the fps_bench intervals.
df_tpu['fps_interval'] = pd.cut(df_tpu.fps_bench, bins=bins, labels=labels, right=True, include_lowest=True)

# For each tracker and each interval, compute the Pearson correlation between HOTA and fps_bench.
# The result will be a Series with a MultiIndex (tracker, fps_interval).
corr_series = df_tpu.groupby(['tracker', 'fps_interval'], observed=False).apply(lambda g: g['hota'].corr(g['fps_bench']), include_groups=False)

# Pivot the Series to create a table with tracker as rows and intervals as columns.
corr_table = corr_series.unstack(level='fps_interval')

# Round correlation values to two decimals.
corr_table = corr_table.round(2)

# Optionally, replace missing values with a dash.
corr_table = corr_table.fillna("-")

# Convert the correlation table to LaTeX.
latex_code = corr_table.to_latex(multicolumn=True, multirow=True, escape=False, float_format="%.2f")

latex_code = latex_code.replace("fps_interval", "Частота")
latex_code = latex_code.replace("tracker", "видеоизображения")

caption = "Корреляция метрики HOTA с частотой видеозображения"
label = "tab:correlation_hota_fpsbench_tracker"
full_latex = (
    "\\begin{table}[htbp]\n"
    f"\\caption{{{caption}}}\n"
    f"\\label{{{label}}}\n"
    "\\centering\n"
    f"{latex_code}\n"
    "\\end{table}"
)


# Save the LaTeX code to a file.
cur_save_path = os.path.join(SAVE_PATH, "correlations")
os.makedirs(cur_save_path, exist_ok=True)
filename = "correlation_hota_fpsbench_tracker.tex"
with open(os.path.join(cur_save_path, filename), "w") as f:
    f.write(full_latex)

print(f"Saved LaTeX table as {filename}")

Saved LaTeX table as correlation_hota_fpsbench_tracker.tex
